In [27]:
import pyodbc
import pygrametl

# Update these with your actual SSMS details
STAGING_CONNECTION_STRING = (
    "Driver={ODBC Driver 17 for SQL Server};"
    "Server=DESKTOP-AR905LJ;"
    "Database=staging;"
    "UID=sa;"
    "PWD=ghom3220;"
)
DWH_CONNECTION_STRING = ( 
    "Driver={ODBC Driver 17 for SQL Server};"
    "Server=DESKTOP-AR905LJ;"
    "Database=DWH;"
    "UID=sa;"
    "PWD=ghom3220;"
)

In [28]:
source_conn = pyodbc.connect(STAGING_CONNECTION_STRING)
source_connection = pygrametl.ConnectionWrapper(source_conn)
source_cursor = source_connection.cursor()
source_cursor.execute("SELECT @@version;")
result = source_cursor.fetchone()
print(f"connected to {result[0]}!")

connected to Microsoft SQL Server 2025 (RTM) - 17.0.1000.7 (X64) 
	Oct 21 2025 12:05:57 
	Copyright (C) 2025 Microsoft Corporation
	Standard Developer Edition (64-bit) on Windows 10 Pro 10.0 <X64> (Build 26200: ) (Hypervisor)
!


In [29]:
dwh_conn = pyodbc.connect(DWH_CONNECTION_STRING)
dwh_connection = pygrametl.ConnectionWrapper(dwh_conn)
dwh_cursor = dwh_connection.cursor()
dwh_cursor.execute("SELECT @@version;")
result = dwh_cursor.fetchone()
print(f"connected to {result[0]}!")

connected to Microsoft SQL Server 2025 (RTM) - 17.0.1000.7 (X64) 
	Oct 21 2025 12:05:57 
	Copyright (C) 2025 Microsoft Corporation
	Standard Developer Edition (64-bit) on Windows 10 Pro 10.0 <X64> (Build 26200: ) (Hypervisor)
!


In [30]:
create_table_sql = """
IF NOT EXISTS (SELECT * FROM sys.objects WHERE object_id = OBJECT_ID(N'[dbo].[Dim_Customer]') AND type in (N'U'))
BEGIN
    CREATE TABLE Dim_Customer (
        Client_ID VARCHAR(50) PRIMARY KEY,
        Full_Name VARCHAR(100),
        Gender VARCHAR(10),
        Age INT,
        Age_Group VARCHAR(20),
        Customer_Type VARCHAR(50),
        registration_date DATE
    );
END
"""
dwh_cursor.execute(create_table_sql)
dwh_connection.commit()

In [35]:
from pygrametl.tables import Dimension
Dim_Customer = pygrametl.tables.Dimension(
    name='Dim_Customer',

    key='Client_ID',

    attributes=[
        'Client_ID',
        'Full_Name',
        'Gender',
        'Age',
        'Age_Group',
        'Customer_Type',
        'registration_date'
    ]
)    

OperationalError: ('08S01', '[08S01] [Microsoft][ODBC Driver 17 for SQL Server]Communication link failure (0) (SQLExecDirectW)')

In [ ]:
source_cursor.execute("SELECT ClientID,NomClient,Sexe,Age,TrancheAge,ClientType,DateInscription FROM source")

for row in source_cursor.fetchall():

    customer = {
        "Client_ID": row.ClientID,
        "Full_Name": row.NomClient,
        "Gender": row.Sexe,
        "Age": row.Age,
        "Age_Group": row.TrancheAge,
        "Customer_Type": row.ClientType,
        "registration_date": row.DateInscription
    }
    Dim_Customer.insert(customer, connection=dwh_connection)

NameError: name 'customerDim' is not defined